In [ ]:
"""
Observer-B proper-time integral zeta_B(O) = integral_0^O tau_o(O') * dzeta/dO' dO'
in Relational Orbital Mechanics (R.O.M.).

Question posed by the blueprint:
    Does zeta_B(O) admit an ELEMENTARY closed form of the shape
        zeta_B(O) = c_S * Sweep(O)  -  f_tension(O)
    with f_tension built from elementary trig, satisfying
        (1) f_tension(0) = f_tension(pi) = 0
        (2) d/dO [c_S*Sweep - f_tension] = tau_o(O) * dzeta/dO  exactly?

This script settles it by inspecting the exact algebraic structure of the
integrand, then quantifies everything numerically for the S2 / Sgr A* system.

ROM definitions used (verified against the skill reference):
    kappa_o^2 = kappa^2 * (1 + e cos O) / (1 - e^2)          (linear in cos O)
    beta_o^2  = kappa_o^2 - beta^2                            (energy invariant)
    tau_o     = sqrt(1 - kappa_o^2) * sqrt(1 - beta_o^2)      (phase spacetime factor)
    dzeta/dO  = (1 - e^2)^(3/2) / (1 + e cos O)^2             (mean-anomaly rate)
    Sweep S(O) = 2*atan( tan(O/2) / sqrt(e_X) ),  e_X = (1+e)/(1-e)
    Tension T(O) = e * e_Y * sin O / (1 + e cos O),  e_Y = sqrt(1 - e^2)
    zeta(O) = S(O) - T(O)
"""

import sympy as sp
import mpmath as mp


# ============================================================
# PART A — SYMBOLIC STRUCTURE OF THE INTEGRAND
# ============================================================
def part_A_symbolic():
    print("=" * 70)
    print("PART A — exact symbolic structure of  tau_o * dzeta/dO")
    print("=" * 70)

    e, b, k, O = sp.symbols('e beta kappa O', positive=True)  # b=beta, k=kappa
    cosO = sp.cos(O)

    # --- ROM phase quantities ---
    kappa_o2 = k**2 * (1 + e * cosO) / (1 - e**2)      # linear in cos O
    beta_o2 = kappa_o2 - b**2
    tau_o2 = (1 - kappa_o2) * (1 - beta_o2)            # = tau_o^2

    # --- KEY IDENTITY: is the radicand a perfect square in cos O ? ---
    # Claim: tau_o^2 = P^2 - (beta^2/2)^2, with P = 1 - kappa_o^2 + beta^2/2 (linear in cos O)
    P = 1 - kappa_o2 + b**2 / 2
    residual = sp.simplify(tau_o2 - (P**2 - (b**2 / 2)**2))
    print("\n[A1] tau_o^2 - [ (1 - kappa_o^2 + beta^2/2)^2 - (beta^2/2)^2 ]  =", residual)
    print("     P = 1 - kappa_o^2 + beta^2/2 is degree", sp.degree(sp.Poly(sp.expand(P), cosO)),
          "in cos O (linear).")
    print("     => radicand under the root = (linear in cosO)^2 - (beta^2/2)^2.")
    print("     A quadratic 'square minus nonzero constant' is NOT a perfect square")
    print("     unless beta = 0.  Hence sqrt(radicand) is irrational in cos O")
    print("     and  integral tau_o dO  is ELLIPTIC, not elementary.")

    # --- Validate e_X, e_Y and the given zeta by reproducing dzeta/dO ---
    e_X = (1 + e) / (1 - e)
    e_Y = sp.sqrt(1 - e**2)
    S = 2 * sp.atan(sp.tan(O / 2) / sp.sqrt(e_X))      # Sweep block
    T = e * e_Y * sp.sin(O) / (1 + e * cosO)           # Tension block
    zeta = S - T
    dzeta_target = (1 - e**2)**sp.Rational(3, 2) / (1 + e * cosO)**2
    check_dzeta = sp.simplify(sp.diff(zeta, O) - dzeta_target)
    print("\n[A2] d/dO[ S(O) - T(O) ] - (1-e^2)^(3/2)/(1+e cosO)^2  =", check_dzeta,
          " (validates e_X, e_Y, zeta)")

    # dS/dO in closed form (needed below)
    dS = sp.simplify(sp.diff(S, O))
    print("     dS/dO =", dS)

    # --- Leading ELEMENTARY antiderivative (drop the -(beta^2/2)^2 term) ---
    # Linearised integrand:  P * dzeta/dO.  Antiderivative claimed:
    #     F0 = (1 + beta^2/2 - kappa^2) * S(O)  -  (1 + beta^2/2) * T(O)
    c_S = 1 + b**2 / 2 - k**2
    c_T = 1 + b**2 / 2
    F0 = c_S * S - c_T * T
    check_F0 = sp.simplify(sp.diff(F0, O) - P * dzeta_target)
    print("\n[A3] d/dO[ (1+beta^2/2-kappa^2) S - (1+beta^2/2) T ] - P*dzeta/dO  =",
          check_F0)
    print("     => the LINEARISED integrand integrates ELEMENTARILY, with")
    print("        Sweep coefficient c_S = 1 + beta^2/2 - kappa^2")
    print("        Tension coefficient c_T = 1 + beta^2/2   (SAME tension block T(O))")

    # --- Is c_S equal to tau = kappa_X*beta_Y (the blueprint's assumed coeff)? ---
    tau_exact = sp.sqrt(1 - k**2) * sp.sqrt(1 - b**2)
    # impose closure kappa^2 = 2 beta^2 and expand both in beta
    c_S_closed = c_S.subs(k**2, 2 * b**2)
    tau_closed = tau_exact.subs(k**2, 2 * b**2)
    diff_series = sp.series(c_S_closed - tau_closed, b, 0, 6).removeO()
    print("\n[A4] Under closure kappa^2 = 2 beta^2:")
    print("     c_S              =", sp.simplify(c_S_closed), " = 1 - (3/2) beta^2 (exact)")
    print("     tau = kappa_X beta_Y series:", sp.series(tau_closed, b, 0, 6))
    print("     c_S - tau  =", diff_series, "  (they differ at O(beta^4))")
    print("     => the blueprint's 'Sweep coefficient = tau' holds only to 1st order.")

    return None


# ============================================================
# PART B — NUMERICS FOR S2 / Sgr A*
# ============================================================
def zeta_closed(e, O):
    """Mean-anomaly zeta(O) closed form on the ascending branch [0, pi]."""
    e_X = (1 + e) / (1 - e)
    e_Y = mp.sqrt(1 - e**2)
    S = 2 * mp.atan(mp.tan(O / 2) / mp.sqrt(e_X))
    T = e * e_Y * mp.sin(O) / (1 + e * mp.cos(O))
    return S - T


def tau_o_num(kappa2, beta2, e, O):
    """Phase spacetime factor tau_o(O) evaluated numerically."""
    kappa_o2 = kappa2 * (1 + e * mp.cos(O)) / (1 - e**2)
    beta_o2 = kappa_o2 - beta2
    return mp.sqrt((1 - kappa_o2) * (1 - beta_o2))


def dzeta_num(e, O):
    return (1 - e**2)**mp.mpf('1.5') / (1 + e * mp.cos(O))**2


def zeta_B_quad(kappa2, beta2, e, O_hi):
    """Exact zeta_B(O_hi) via high-precision quadrature of tau_o * dzeta/dO."""
    integrand = lambda O: tau_o_num(kappa2, beta2, e, O) * dzeta_num(e, O)
    return mp.quad(integrand, [0, O_hi])


def zeta_B_elementary(kappa2, beta2, e, O):
    """Leading ELEMENTARY closed form F0(O) = c_S*S - c_T*T (drops -(beta^2/2)^2)."""
    e_X = (1 + e) / (1 - e)
    e_Y = mp.sqrt(1 - e**2)
    S = 2 * mp.atan(mp.tan(O / 2) / mp.sqrt(e_X))
    T = e * e_Y * mp.sin(O) / (1 + e * mp.cos(O))
    c_S = 1 + beta2 / 2 - kappa2
    c_T = 1 + beta2 / 2
    return c_S * S - c_T * T


def part_B_numeric():
    mp.mp.dps = 50
    print("\n" + "=" * 70)
    print("PART B — S2 / Sgr A* numbers")
    print("=" * 70)

    # --- Physical constants and S2 orbital parameters (literature) ---
    G = mp.mpf('6.67430e-11')
    c = mp.mpf('299792458')
    Msun = mp.mpf('1.98892e30')
    yr = mp.mpf('31557600')            # Julian year (s)

    e = mp.mpf('0.884649')             # S2 eccentricity (Gravity Collab.)
    M = mp.mpf('4.297e6') * Msun       # Sgr A* mass
    T = mp.mpf('16.0455') * yr         # S2 orbital period

    R_s = 2 * G * M / c**2
    beta = (mp.pi * R_s / (T * c))**(mp.mpf(1) / 3)   # global kinetic projection
    beta2 = beta**2
    kappa2 = 2 * beta2                                 # closure kappa^2 = 2 beta^2
    tau = mp.sqrt(1 - kappa2) * mp.sqrt(1 - beta2)     # kappa_X * beta_Y

    print(f"  e        = {e}")
    print(f"  R_s      = {mp.nstr(R_s, 8)} m")
    print(f"  beta     = {mp.nstr(beta, 10)}   beta^2 = {mp.nstr(beta2, 6)}")
    print(f"  kappa^2  = {mp.nstr(kappa2, 6)}   (=2 beta^2, closure)")
    print(f"  beta_p   = {mp.nstr(beta*mp.sqrt((1+e)/(1-e)), 8)}  (perihelion speed / c)")
    print(f"  tau      = {mp.nstr(tau, 12)}")
    print(f"  (beta^2/2)^2 obstruction = {mp.nstr((beta2/2)**2, 6)}  (drives elliptic term)")

    O_hi = mp.pi / 2
    A_quoted = mp.mpf('5850748.82693')     # observer-A answer given by user

    # scale factor R_s/(2 beta^3 c) = T/(2 pi)
    prefactor = R_s / (2 * beta**3 * c)
    print(f"\n  R_s/(2 beta^3 c) = {mp.nstr(prefactor, 10)} s   (= T/2pi = {mp.nstr(T/(2*mp.pi),10)} s)")

    # --- observer A cross-check ---
    zeta_A = zeta_closed(e, O_hi)
    A_pred = prefactor * zeta_A
    print(f"\n  zeta(pi/2)        = {mp.nstr(zeta_A, 12)}")
    print(f"  A predicted       = {mp.nstr(A_pred, 12)} s")
    print(f"  A quoted          = {A_quoted} s")
    print(f"  (ratio A_pred/A_quoted = {mp.nstr(A_pred/A_quoted, 8)} -> params match if ~1)")

    # --- observer B ---
    zeta_B_exact = zeta_B_quad(kappa2, beta2, e, O_hi)     # elliptic-exact via quadrature
    zeta_B_elem = zeta_B_elementary(kappa2, beta2, e, O_hi)  # elementary leading form

    ratio_exact = zeta_B_exact / zeta_A       # scale-free: B_time/A_time
    ratio_elem = zeta_B_elem / zeta_A

    B_time_exact = ratio_exact * A_quoted     # anchor to user's A number
    B_time_elem = ratio_elem * A_quoted

    print(f"\n  zeta_B(pi/2) exact (quad)     = {mp.nstr(zeta_B_exact, 14)}")
    print(f"  zeta_B(pi/2) elementary F0    = {mp.nstr(zeta_B_elem, 14)}")
    print(f"  difference (exact - elem)     = {mp.nstr(zeta_B_exact - zeta_B_elem, 6)}")

    print(f"\n  B time (exact)      = {mp.nstr(B_time_exact, 12)} s")
    print(f"  B time (elementary) = {mp.nstr(B_time_elem, 12)} s")
    print(f"  elementary-form error = {mp.nstr(B_time_exact - B_time_elem, 6)} s")
    print(f"\n  A - B (proper-time deficit over 0->pi/2) = {mp.nstr(A_quoted - B_time_exact, 8)} s")
    print(f"  fractional deficit (1 - B/A)             = {mp.nstr(1 - ratio_exact, 6)}")

    # --- Test the blueprint's global boundary condition zeta_B(2pi) = 2 pi tau ---
    # equivalently <tau_o>_time over one orbit = tau ?
    full = mp.quad(lambda O: tau_o_num(kappa2, beta2, e, O) * dzeta_num(e, O), [0, mp.pi, 2*mp.pi])
    tau_o_mean = full / (2 * mp.pi)
    c_S_val = 1 + beta2 / 2 - kappa2
    print(f"\n  <tau_o>_time = zeta_B(2pi)/2pi = {mp.nstr(tau_o_mean, 14)}")
    print(f"  tau = kappa_X beta_Y           = {mp.nstr(tau, 14)}")
    print(f"  c_S = 1+beta^2/2-kappa^2       = {mp.nstr(c_S_val, 14)}")
    print(f"  <tau_o> - tau  = {mp.nstr(tau_o_mean - tau, 6)}   (nonzero => f_tension(pi)!=0 under tau-ansatz)")
    print(f"  <tau_o> - c_S  = {mp.nstr(tau_o_mean - c_S_val, 6)}")

    return None


if __name__ == '__main__':
    part_A_symbolic()
    part_B_numeric()

PART A — exact symbolic structure of  tau_o * dzeta/dO

[A1] tau_o^2 - [ (1 - kappa_o^2 + beta^2/2)^2 - (beta^2/2)^2 ]  = 0
     P = 1 - kappa_o^2 + beta^2/2 is degree 1 in cos O (linear).
     => radicand under the root = (linear in cosO)^2 - (beta^2/2)^2.
     A quadratic 'square minus nonzero constant' is NOT a perfect square
     unless beta = 0.  Hence sqrt(radicand) is irrational in cos O
     and  integral tau_o dO  is ELLIPTIC, not elementary.

[A2] d/dO[ S(O) - T(O) ] - (1-e^2)^(3/2)/(1+e cosO)^2  = (-2*e*sqrt(-1/(e - 1))*sqrt(1 - e**2)*sin(O/2)**2 + e*sqrt(-1/(e - 1))*sqrt(1 - e**2) - e*sqrt(e + 1)*cos(O) + sqrt(-1/(e - 1))*sqrt(1 - e**2) - sqrt(e + 1))/(sqrt(-1/(e - 1))*(-2*e**2*cos(O/2)**2*cos(O) + e**2*cos(O) - 2*e*cos(O/2)**2 - e*cos(O) + e - 1))  (validates e_X, e_Y, zeta)
     dS/dO = sqrt(e + 1)/(sqrt(-1/(e - 1))*(2*e*cos(O/2)**2 - e + 1))

[A3] d/dO[ (1+beta^2/2-kappa^2) S - (1+beta^2/2) T ] - P*dzeta/dO  = (e**2*sqrt(-1/(e - 1))*(1 - e**2)**(3/2)*(beta**2 + 2)*(-e + 